In [ ]:
def bursting_metrics(
    sorting_clean,
    recording_clean,
    *,
    bin_size_s: float = 0.050,        # pour Fano factor / spike counts
    refractory_s: float = 0.002,      # pour frac ISI < refractory
    min_spikes_for_stats: int = 10,   # éviter des stats instables
):
    """
    Calcule des métriques standard et robustes (Elephant) par unité à partir d'un
    sorting SpikeInterface (frames) déjà remappé sur un recording concaténé.

    Paramètres
    ----------
    sorting_clean : spikeinterface Sorting
        Produit par remap_sorting_to_concat(...).
    recording_clean : spikeinterface Recording
        Correspond à `recording_e_clean` (concat des good chunks), et filtré.
        Sert à obtenir la durée (t_stop) et la sampling frequency.
    bin_size_s : float
        Taille de bin (s) pour les métriques basées sur comptage (Fano).
    refractory_s : float
        Seuil (s) pour calculer frac(ISI < refractory).
    min_spikes_for_stats : int
        Si < min spikes, certaines métriques Elephant retournent NaN par design.

    Retour
    ------
    df : pandas.DataFrame
        Une ligne par unité, avec métriques:
        - firing_rate_hz, num_spikes, duration_s
        - isi_mean_s, isi_median_s, isi_min_s, isi_max_s
        - cv_isi (classique), cv2 (robuste non-stationnarité), lv (robuste)
        - frac_isi_lt_refractory
        - fano_factor (sur bins de bin_size_s)
    """
    import quantities as pq
    import neo
    from elephant.statistics import isi as elephant_isi
    from elephant.statistics import cv2 as elephant_cv2
    from elephant.statistics import lv as elephant_lv
    from elephant.statistics import time_histogram
    from elephant.statistics import fanofactor

    # --- durée et FS ---
    fs = float(recording_clean.get_sampling_frequency())
    n_frames = int(recording_clean.get_num_frames())
    duration_s = n_frames / fs

    t_start = 0.0 * pq.s
    t_stop = duration_s * pq.s
    bin_size = float(bin_size_s) * pq.s

    rows = []

    for unit_id in sorting_clean.unit_ids:
        # SpikeInterface donne des frames (int). Convertir en secondes.
        st_frames = sorting_clean.get_unit_spike_train(unit_id=unit_id)  # frames
        st_frames = np.asarray(st_frames, dtype=np.int64)

        if st_frames.size == 0:
            rows.append({
                "unit_id": unit_id,
                "num_spikes": 0,
                "duration_s": duration_s,
                "firing_rate_hz": 0.0,
                "isi_mean_s": np.nan,
                "isi_median_s": np.nan,
                "isi_min_s": np.nan,
                "isi_max_s": np.nan,
                "cv_isi": np.nan,
                "cv2": np.nan,
                "lv": np.nan,
                "frac_isi_lt_refractory": np.nan,
                "fano_factor": np.nan,
            })
            continue

        st_sec = st_frames / fs

        # Neo SpikeTrain (unité + bornes temporelles)
        spiketrain = neo.SpikeTrain(st_sec * pq.s, t_start=t_start, t_stop=t_stop)

        # ---- Métriques simples ----
        n_spikes = int(spiketrain.size)
        fr_hz = n_spikes / duration_s if duration_s > 0 else np.nan

        # ISI Elephant (renvoie Quantity array)
        if n_spikes >= 2:
            isi_q = elephant_isi(spiketrain)  # quantities
            isi_s = isi_q.rescale(pq.s).magnitude

            isi_mean = float(np.mean(isi_s))
            isi_median = float(np.median(isi_s))
            isi_min = float(np.min(isi_s))
            isi_max = float(np.max(isi_s))

            # CV classique (moins robuste si non-stationnaire)
            cv_isi = float(np.std(isi_s, ddof=1) / np.mean(isi_s)) if isi_s.size > 1 and np.mean(isi_s) > 0 else np.nan

            frac_refr = float(np.mean(isi_s < float(refractory_s))) if isi_s.size > 0 else np.nan
        else:
            isi_mean = isi_median = isi_min = isi_max = np.nan
            cv_isi = np.nan
            frac_refr = np.nan

        # ---- Métriques robustes Elephant ----
        # CV2 et LV sont définies sur ISI locaux et sont moins sensibles à un drift lent du taux.
        if n_spikes >= min_spikes_for_stats:
            try:
                cv2_val = float(elephant_cv2(spiketrain))
            except Exception:
                cv2_val = np.nan
            try:
                lv_val = float(elephant_lv(spiketrain))
            except Exception:
                lv_val = np.nan
        else:
            cv2_val = np.nan
            lv_val = np.nan

        # ---- Fano factor (variabilité des counts entre bins) ----
        # On fait un histogramme de spikes binned puis Fano factor.
        # Attention: si trop peu de bins non vides => NaN possible.
        try:
            counts = time_histogram([spiketrain], bin_size=bin_size, output="counts")
            # counts est un neo.AnalogSignal avec shape (n_bins, n_trains)
            # On prend la colonne 0.
            fano = float(fanofactor(counts[:, 0]))
        except Exception:
            fano = np.nan

        rows.append({
            "unit_id": unit_id,
            "num_spikes": n_spikes,
            "duration_s": duration_s,
            "firing_rate_hz": float(fr_hz),
            "isi_mean_s": isi_mean,
            "isi_median_s": isi_median,
            "isi_min_s": isi_min,
            "isi_max_s": isi_max,
            "cv_isi": cv_isi,
            "cv2": cv2_val,
            "lv": lv_val,
            "frac_isi_lt_refractory": frac_refr,
            "fano_factor": fano,
        })

    df = pd.DataFrame(rows).set_index("unit_id", drop=True)
    return df
